# Dry Machina — Parametric Design & CAM Compiler Tutorial

Welcome to **Dry Machina (`dry`)**, a deterministic parametric design and CAM compiler.

This interactive notebook demonstrates:
1. **Continuous Helical Vase Toolpath** generation with S-curve acceleration.
2. **TPMS (Triply Periodic Minimal Surface)** implicit field slicing (Gyroid & Schwarz Diamond).
3. **5-Axis Surface Normal Draping** with spatial normal vectors.
4. **CNC Rectangular Pocket Milling** with tool-radius compensation.
5. **Physical Machine Pre-Flight Safety Verification** against industrial machine envelopes.

In [ ]:
# Install and import Dry Machina Python SDK
!pip install --quiet dry-machina
import dry
print(f"Dry version: {dry.__version__}")

## 1. Continuous Helical Vase
Authoring continuous toolpaths directly in mathematical space without STL mesh generation.

In [ ]:
import math

d = dry.Design("helical_vase")
d.speed(print_speed=1800.0, travel_speed=6000.0)
d.extruder(True)

layers = 50
layer_height = 0.2
radius = 30.0
segments_per_rev = 72

for layer in range(layers):
    z = layer * layer_height
    for i in range(segments_per_rev):
        theta = (i / segments_per_rev) * 2.0 * math.pi
        # Helical climb
        curr_z = z + (i / segments_per_rev) * layer_height
        curr_r = radius + 3.0 * math.sin(6.0 * theta + layer * 0.1)
        x = 100.0 + curr_r * math.cos(theta)
        y = 100.0 + curr_r * math.sin(theta)
        d.move(x=x, y=y, z=curr_z)

gcode = d.gcode(flavor="marlin")
print(f"Generated {len(gcode.splitlines())} lines of continuous helical G-code.")

## 2. TPMS Metamaterial Lattice (Gyroid)
Generate functional implicit surface metamaterials directly evaluated into toolpaths.

In [ ]:
tpms_ops = dry.generate_tpms_ops(
    surface="gyroid",
    bounds_min=(0.0, 0.0, 0.0),
    bounds_max=(40.0, 40.0, 20.0),
    cell_size=(10.0, 10.0, 10.0),
    isovalue=0.0,
    layer_height=0.4,
    resolution=40
)
print(f"Generated {len(tpms_ops)} TPMS toolpath operations.")

## 3. In-Process Machine Safety Verification
Verify toolpaths against physical build volume limits, feedrate caps, and thermal envelopes.

In [ ]:
catalog = dry.MachineCatalog.default_catalog()
bambu = catalog.get("bambu_x1c")

# Perform pre-flight verification on the design
report = d.verify()
print(f"Safety findings: {len(report.findings)}")
if len(report.findings) == 0:
    print("✅ 100% Machine Safe: Ready for physical execution.")
else:
    for f in report.findings:
        print(f"⚠️ [{f.severity}] {f.message}")